# Testes via ollama

In [1]:
import ollama

In [2]:
MODEL = "gemma3:12b"

In [3]:
def query_llm(model: str, system_prompt: str, user_prompt: str):

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options={
            "temperature": 0, # provisório, para garantir respostas consistentes
            "seed": 2026
        }
    )
    return response["message"]["content"]


In [4]:
a = query_llm(MODEL, "", "What is 5W1H event scheme?")

In [5]:
print(a)

The 5W1H event scheme is a simple yet powerful framework for planning and organizing events (and really, any project or task). It's based on answering six key questions, each represented by a "W" or "H." It ensures you consider all the essential aspects before, during, and after the event.

Here's a breakdown of each element:

**1. Who? (Attendees & Key People)**

*   **Who is the event for?** (Target audience - demographics, interests, needs)
*   **Who are the speakers/performers/presenters?** (Identify and confirm their participation)
*   **Who is responsible for what?** (Assign roles and responsibilities to team members - planning, logistics, marketing, etc.)
*   **Who needs to be informed?** (Stakeholders, sponsors, vendors)

**2. What? (The Event Itself)**

*   **What is the purpose of the event?** (Goals, objectives - what do you want to achieve?)
*   **What is the event about?** (Theme, content, activities, agenda)
*   **What is the format of the event?** (Conference, workshop, 

# Construção de prompts de sistema e usuário

In [6]:
import datetime

In [42]:
def get_system_prompt() -> str:
    return (
            "You are an event generator that works with a 5W1H event structure.\n\n"
            "Each event has the following JSON structure:\n"
            "{\n"
            '  "event_id": "unique_string_identifier_based_on_content_or_date",\n'
            # '  "parent_event_id": "id_of_parent_event_or_null",\n'
            '  "what": String - "short description of the event",\n'
            '  "when": String - "ISO date in the format YYYY-MM-DD",\n'
            '  "where": List[Dict] {\n'
            '      "text": String - "textual description of the spatial scope (e.g., local, national, global)",\n'
            '      "locations": List[Dict] - "possible *countries* involved and their locations" [\n'
            "          {\n"
            '              "country": "country name (never use generic words like \'global\', \'world\', \'planet\', or \'earth\')",\n'
            '              "lat": latitude_or_null,\n'
            '              "long": longitude_or_null\n'
            "          },\n"
            '          "..." \n'
            "      ]\n"
            "  },\n"
            '  "who": String - "main actors involved",\n'
            '  "why": String - "motivation or cause",\n'
            '  "how": String - "how the event happens or is executed",\n'
            "}\n\n"
            "Rules for WHERE:\n"
            '- The field "where.text" summarizes the spatial scope '
            '(e.g., "Global impact across several countries").\n'
            '- The list "where.locations" MUST contain *at least one* concrete location.\n'
            "- For events that are global or affect many regions, you MUST select a SMALL LIST "
            "(2 to 5) of representative countries and/or cities as concrete locations.\n"
            '- Never use generic values such as "global", "world", "planet", or "earth" as the country name.\n'
            '- Each entry in "where.locations" MUST be a real or realistic country/city pair '
            "with lat/long whenever possible.\n\n"
            "Language:\n"
            "- The text fields (what, where.text, who, why, how) MUST be written in the same language "
            "as the news article.\n\n"
            "Output:\n"
            "- You MUST return a single JSON object that strictly follows the JSON schema provided by the system.\n"
            "- Do not include any explanations, comments, or markdown fences in your output.\n"
            "- Return ONLY JSON.\n"
        )

In [8]:
def get_user_prompt(title: str, description: str, url: str, published: datetime) -> str:
    return (
            "Given the following news article, extract EXACTLY ONE event in the 5W1H format.\n\n"
            "Use the schema described by the system "
            "(event_id, parent_event_id, what, when, where, who, why, how, category)\n"
            "and strictly follow the rules for the \"where\" field (spatial scope and list of locations).\n\n"
            f"Title: {title}\n"
            f"Description: {description}\n"
            f"URL: {url}\n"
            f"Publication date: {published.isoformat()}\n\n"
            "All textual fields of the event (what, where.text, who, why, how) MUST be written "
            "in the SAME LANGUAGE as the article above.\n"
            "Return only the JSON object for this single event, with no explanations or additional text."
        )

# Carregando notícias coletadas

In [1]:
import pickle
import pandas as pd

In [2]:
raw_news = pd.read_pickle("../raw/gnews_results_mar_abril_ai_2026.pkl")
news_df = pd.DataFrame(raw_news)

In [3]:
raw_news

[{'title': 'U.S. Postal Inspectors Warn Customers to Avoid Scams that Use Artificial Intelligence - PR Newswire',
  'description': 'U.S. Postal Inspectors Warn Customers to Avoid Scams that Use Artificial Intelligence  PR Newswire',
  'published date': 'Sun, 01 Mar 2026 08:00:00 GMT',
  'url': 'https://news.google.com/rss/articles/CBMi1gFBVV95cUxQZnhvVHFTLS1Iai1PakIyekpjYmZXUG5mbzgwNFNFTzY5YTc0cm15U0NETTRlYmlpRS1ZQnVVbFFRVzVOTHhacVlPVWVwUDlGR251bmdNejFONEpwNkRERHQ5clNqekIzY0N4RVpObTQtR01vWjczUVBUSGZTSUhUNHVsTDdfWURiQndQRUZfam5Ba2lVU3JZNVAydUlrWEJxcHN0S0FMZHpWb0V3YTVZaHJKNmRURmlsTFpYRVdndkVQZ1kzSXczSHRYeFZUVXZLck5rSS1R?oc=5&hl=en-US&gl=US&ceid=US:en',
  'publisher': {'href': 'https://www.prnewswire.com', 'title': 'PR Newswire'},
  'keyword': 'artificial intelligence'},
 {'title': 'Honor Week panel discusses the future of artificial intelligence in academic integrity - The Cavalier Daily',
  'description': 'Honor Week panel discusses the future of artificial intelligence in academic inte

In [11]:
# remoção de duplicatas
news_df = news_df.drop_duplicates(subset=["title"])

In [12]:
# amostrando 10 notícias para teste
sample_news = news_df.sample(n=10, random_state=2026)

# Testando extração de componentes 5W1H

In [13]:
from tqdm.notebook import tqdm

In [15]:
news_df

,title,description,published date,url,publisher,keyword
0,U.S. Postal Inspectors Warn Customers to Avoid...,U.S. Postal Inspectors Warn Customers to Avoid...,"Sun, 01 Mar 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMi1gFBV...,"{'href': 'https://www.prnewswire.com', 'title'...",artificial intelligence
1,Honor Week panel discusses the future of artif...,Honor Week panel discusses the future of artif...,"Sun, 01 Mar 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMizAFBV...,"{'href': 'https://www.cavalierdaily.com', 'tit...",artificial intelligence
2,Decoding the language of sleep with artificial...,Decoding the language of sleep with artificial...,"Sat, 28 Feb 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMiiwFBV...,"{'href': 'https://www.thelancet.com', 'title':...",artificial intelligence
3,2 Top Artificial Intelligence Stocks to Buy in...,2 Top Artificial Intelligence Stocks to Buy in...,"Sun, 01 Mar 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMiigFBV...,"{'href': 'https://finance.yahoo.com', 'title':...",artificial intelligence
4,Artificial intelligence versus traditional app...,Artificial intelligence versus traditional app...,"Sun, 01 Mar 2026 08:00:00 GMT",https://news.google.com/rss/articles/CBMiX0FVX...,"{'href': 'https://www.nature.com', 'title': 'N...",artificial intelligence
...,...,...,...,...,...,...
26666,MSS backs 20 Korean AI startups in overseas ex...,MSS backs 20 Korean AI startups in overseas ex...,"Mon, 30 Mar 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMiggFBV...,"{'href': 'https://biz.chosun.com', 'title': 'C...",AI investment
26667,Where to invest during the Iran War - Morningstar,Where to invest during the Iran War Morningstar,"Mon, 30 Mar 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMidkFVX...,"{'href': 'https://www.morningstar.com.au', 'ti...",AI investment
26668,Amazon.com Inc Stock Faces Pressure from $200 ...,Amazon.com Inc Stock Faces Pressure from $200 ...,"Mon, 30 Mar 2026 07:00:00 GMT",https://news.google.com/rss/articles/CBMiwgFBV...,"{'href': 'https://www.ad-hoc-news.de', 'title'...",AI investment
26669,Japan’s JX Metals Ramps Up Investment Amid Ris...,Japan’s JX Metals Ramps Up Investment Amid Ris...,"Mon, 30 Mar 2026 06:05:45 GMT",https://news.google.com/rss/articles/CBMinAFBV...,"{'href': 'https://www.cxodigitalpulse.com', 't...",AI investment


In [19]:
system_prompt = get_system_prompt()
responses = []
for i, row in tqdm(sample_news.iterrows(), total=sample_news.shape[0], desc="- Extracting events components"):
    user_prompt = get_user_prompt(
        title=row["title"],
        description=row["description"],
        url=row["url"],
        published=pd.to_datetime(row["published date"])
    )
    response = query_llm(MODEL, system_prompt, user_prompt)
    responses.append(response)

- Extracting events components:   0%|          | 0/10 [00:00<?, ?it/s]

# Processando saídas do LLM

In [20]:
import json

In [36]:
def decode_response(response: str):
    # remover possíveis markdown fences e a palavra "json" que o modelo pode incluir
    response = response.strip("`").replace("json", "")
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Response was: {response}")
        return None

In [24]:
print(responses[1])

```json
{
  "event_id": "a1b2c3d4e5f6g7h8i9j0",
  "parent_event_id": null,
  "what": "EU lawmakers advance revisions to the AI Act.",
  "when": "2026-03-13",
  "where": {
    "text": "European Union",
    "locations": [
      {
        "country": "Belgium",
        "lat": 50.8504,
        "long": 4.9131
      },
      {
        "country": "Germany",
        "lat": 51.1657,
        "long": 10.4515
      }
    ]
  },
  "who": "EU lawmakers",
  "why": "To refine and improve the AI Act.",
  "how": "Through legislative processes and revisions."
}
```


In [37]:
parsed_events = [decode_response(r) for r in responses]

# Visualizando e analisando a qualidade da extração

In [41]:
for source, event in zip(sample_news["title"], parsed_events):
    print(f"Source: {source}")
    print(f"Extracted event:")
    for key, value in event.items():
        print(f"\t - {key}: {value}")
    print("-" * 80)

Source: Microsoft is considering legal action against OpenAI and Amazon - Techzine Global
Extracted event:
	 - event_id: microsoft_openai_amazon_legal_20260318
	 - parent_event_id: None
	 - what: Microsoft is considering legal action against OpenAI and Amazon.
	 - when: 2026-03-18
	 - where: {'text': 'Legal considerations impacting companies in the United States.', 'locations': [{'country': 'United States', 'lat': 37.0902, 'long': -95.7129}]}
	 - who: Microsoft, OpenAI, Amazon
	 - why: Unspecified reasons for potential legal action.
	 - how: Through legal proceedings, details yet to be disclosed.
--------------------------------------------------------------------------------
Source: EU lawmakers move forward on AI Act changes - Digital Watch Observatory
Extracted event:
	 - event_id: a1b2c3d4e5f6g7h8i9j0
	 - parent_event_id: None
	 - what: EU lawmakers advance revisions to the AI Act.
	 - when: 2026-03-13
	 - where: {'text': 'European Union', 'locations': [{'country': 'Belgium', 'lat'